# 1. Use Sentence-BERT (SBERT) or MiniLM for Semantic Search

# Handles synonyms, context, paraphrasing better than TF-IDF or BM25.

In [3]:
# %pip install sentence_transformers

In [5]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
import sys
import os
import json 
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ENV = 'dev'

# Load config
with open("config.json") as f:
    config = json.load(f)

if ENV == 'dev':
    base_path = config[f"{ENV}_path"]  
    data_path = os.path.join(base_path, "data")
    model_path = os.path.join(base_path, "models")
    print("Base path:", base_path)    
    print("Data path:", data_path)
    print("Model path:", model_path)

Base path: /Users/jillchow/HBS/hbs_search_engine
Data path: /Users/jillchow/HBS/hbs_search_engine/data
Model path: /Users/jillchow/HBS/hbs_search_engine/models


In [8]:
queryfile_name = "query.csv" 
queryfile_path = os.path.join(data_path, queryfile_name)
productfile_name = "product.csv" 
productfile_path = os.path.join(data_path, productfile_name)
labelfile_name = "label.csv" 
labelfile_path = os.path.join(data_path, labelfile_name)
query_df = pd.read_csv(queryfile_path, sep='\t')
product_df = pd.read_csv(productfile_path, sep='\t')
label_df = pd.read_csv(labelfile_path, sep='\t')

# print('query_df: search queries')
# display(query_df.head()) # watch for null values in query class column 
# query_df.info()

# print('\n product_df: product information')
# display(product_df.head())
# product_df.info() # watch for null values other than product_id, product_name, product_features

# print('\n label_df: ground truth labels')
# display(label_df.head())
# label_df.info()

In [ ]:
# product_corpus = (product_df['product_name'] + ' ' + product_df['product_description']).fillna("").astype(str).tolist()
# product_embeddings = model.encode(product_corpus, convert_to_tensor=True)



In [21]:
def get_top_products_sbert(query, top_n=10):
    query_embedding = model.encode(query, convert_to_tensor=True)
    scores = util.cos_sim(query_embedding, product_embeddings)[0]
    scores = scores.cpu()
    top_indices = scores.argsort(descending=True)[:top_n]
    return top_indices.numpy()

# update to use the label_df, and add to the parameters
def get_exact_matches_for_query(query_id, label_df):
    grouped_label_df = label_df.groupby('query_id')
    query_group = grouped_label_df.get_group(query_id)
    exact_matches = query_group.loc[query_group['label'] == 'Exact']['product_id'].values
    return exact_matches

In [18]:
import importlib
import helper
importlib.reload(helper)
from helper import calculate_tfidf, get_top_products, map_at_k, get_top_product_ids_for_query, get_exact_matches_for_query

In [ ]:
query_df['top_product_ids'] = query_df['query'].apply(lambda q: product_df.iloc[get_top_products_sbert(q)].product_id.tolist())

# adding the list of exact match product_IDs from labels_df
query_df['relevant_ids'] = query_df['query_id'].apply(
      lambda qid: get_exact_matches_for_query(qid, label_df)
)



In [24]:
query_df['map@k'] = query_df.apply(lambda x: map_at_k(x['relevant_ids'], x['top_product_ids'], k=10), axis=1)
print("🧼 SBERT MAP@10:", query_df['map@k'].mean())

🧼 SBERT MAP@10: 0.3569116650132275


In [33]:
# check how many queries have exact matches for each query_id


exact_matches = label_df[label_df['label'] == 'Exact']

query_id_counts = exact_matches.groupby('query_id').size().reset_index(name='exact_match_count')
query_id_counts['exact_match_count'].describe()

count    379.000000
mean      67.583113
std      114.687061
min        1.000000
25%        4.000000
50%       28.000000
75%       81.000000
max      878.000000
Name: exact_match_count, dtype: float64

In [35]:
## lets try cross encoder

# %pip install -U sentence-transformers

In [36]:
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')  # very fast and strong


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.66k [00:00<?, ?B/s]